# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step example for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains multiple record sets, fields, and columns suitable for analysis using `mlcroissant`.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nIdentifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Spatial Coverage: {metadata.spatialCoverage}")
print(f"Temporal Coverage: {metadata.temporalCoverage}")

# If data limitations or biases are present, print first one
if hasattr(metadata, 'dataLimitations'):
    print(f"Data Limitation (example): {metadata.dataLimitations[0]}")
if hasattr(metadata, 'dataBiases'):
    print(f"Data Bias (example): {metadata.dataBiases[0]}")

## 2. Data Overview
Explore the dataset's available record sets, and fields, displaying all entities by their `@id` fields as recommended by the Croissant specification and the `mlcroissant` API.

In [ ]:
# List record sets and their field @ids

record_sets = dataset.record_sets

print(f"Number of record sets: {len(record_sets)}\n")
for i, record_set in enumerate(record_sets):
    print(f"Record Set {i+1}: @id = {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '(none)')}")
    if 'description' in record_set:
        print(f"  Description: {record_set['description']}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields: {', '.join([f['@id'] for f in fields]) if fields else '(none)'}")
    if 'column' in record_set:
        columns = record_set.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        print(f"  Columns: {', '.join([c['@id'] for c in columns]) if columns else '(none)'}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for further exploration. All references use each entity's unique `@id` to ensure consistent data access.

We'll illustrate this with the first available record set. Use the record set and field `@id`s from above.

In [ ]:
# Extract each record set by @id into pandas DataFrames
dataframes = {}

# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded record set: {record_set_id}, with {df.shape[0]} rows, {df.shape[1]} columns.")
        else:
            print(f"Record set {record_set_id} contains no records.")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# List columns from the first populated record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No DataFrames loaded. Check the schema or dataset availability.")

## 4. Exploratory Data Analysis (EDA)
Apply some common EDA steps: filter records using a numeric field, normalize it, and optionally group by a categorical field. All field accesses reference the proper `@id` as per Croissant conventions.

In [ ]:
# We pick the first available DataFrame and investigate its numeric columns by @id
if dataframes:
    import numpy as np
    rs_id = first_rs_id
    df = dataframes[rs_id].copy()

    # Identify numeric columns using pandas dtype inspection
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric columns (by @id) in record set {rs_id}: {numeric_cols}")

    if numeric_cols:
        numeric_field = numeric_cols[0]  # Use the first numeric field's @id
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean (={threshold:.3f}):")
        display(filtered_df.head())

        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Try to group by a likely categorical column, e.g. 'gender' or first object type column
        candidate_group_fields = df.select_dtypes(exclude=[np.number]).columns.tolist()
        group_field = None
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            print(f"\nGrouping statistics by field '{group_field}' (@id):")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric fields to analyze.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize a numeric field's distribution or the relationship between two fields, always referencing their `@id`. This demonstrates how to quickly explore the data visually.

In [ ]:
# Quick visualization with matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No suitable data available for plotting.")

## 6. Conclusion
In this notebook, we've used the `mlcroissant` library to:
- Load the FAIR² dataset from its Croissant schema using its URL
- Examine available record sets, fields, and their Croissant `@id` values
- Extract and load data with fully qualified @id references, ensuring robust reproducibility
- Perform exploratory data analysis, including filtering and normalization
- Visualize the distribution of key numeric fields and group statistics by categorical attributes

This workflow illustrates how Croissant and `mlcroissant` enable programmatic, FAIR-compliant exploration of machine learning datasets.